# SuperKernel 课后实践

通过前面章节的学习，我们已经了解了自定义算子接入 SuperKernel 图融合的基本方法。为了巩固所学知识，现提供以下实践练习：

为已有的 int8 GELU 自定义算子增加 SuperKernel 适配，将其接入网络，开启融合优化并完成运行验证。基础工程已提供 GELU 计算逻辑、普通 kernel 直调、PyTorch 注册及运行脚本。

本作业提供的样例中的主要算子如下：

| 顺序 | 算子 | KernelType | 说明 |
| --- | --- | --- | --- |
| 1 | `GroupedMatmul_pre` | `AIC_ONLY` | 前段矩阵计算，输出 `int32` |
| 2 | `DequantSwigluQuant` | `AIV_ONLY` | 反量化、激活与量化融合计算，输出 `int8` |
| 3 | `GELU` | `AIV_ONLY` | 自定义gelu算子，输出 `bfloat16` |
| 4 | `GroupedMatmul_after` | `MIX_1_1` | 后段矩阵计算，输出 `bfloat16` |
| 5 | `DynamicQuantV2` | `AIV_ONLY` | 将层输出动态量化为 `int8` |

**计算公式**：

GELU 的近似计算公式为：

$$GELU(x) \approx 0.5x\left[1+\tanh\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]$$

本题源码采用以下形式进行向量计算：

$$GELU(x) \approx \frac{x}{1+\exp[-1.595769(x+0.044715x^3)]}$$

输入先转换为浮点类型完成计算，再按照源码的类型转换规则转回 int8。

**要求**：

- GELU 输入、输出均为连续的 `int8[9216, 2048]` 张量；网络输入、最终输出均为 `int8[9216, 7168]`。
- 为 GELU 增加参数结构体和 SK 入口，通过 `SK_BIND` 绑定普通入口与 SK 入口，保留普通 `<<<>>>` 直调及 PyTorch 注册。
- 在 `DequantSwigluQuant` 与第二个 `GroupedMatmul` 之间接入 GELU，使用 `npugraph_ex` 开启静态 Kernel 编译和 SuperKernel 优化。
- 对比 SK 关闭、开启时的输出，结合 Profiling 和融合编译产物判断 GELU 是否参与融合、分析性能变化。
- 默认编译架构为 `dav-2201`；其他架构通过 `SK_NPU_ARCHS` 指定并在对应硬件验证。

---

环境准备：

请在原仓库的 `06_advanced_features` 目录启动 notebook，确保当前 Python 内核和运行脚本使用的 `python3` 均能导入 `torch`、`torch_npu`。执行以下单元格加载 CANN 环境，并按实际空闲设备修改 `DEVICE_ID`。


In [ ]:
import os
import subprocess
import shlex
from pathlib import Path


def find_cann_home():
    candidates = []
    for key in ["ASCEND_HOME_PATH", "ASCEND_TOOLKIT_HOME"]:
        value = os.environ.get(key)
        if value:
            candidates.append(Path(value).expanduser())

    candidates.extend([
        Path.home() / "Ascend/cann",
        Path.home() / "Ascend/ascend-toolkit/latest",
        Path("/usr/local/Ascend/cann"),
        Path("/usr/local/Ascend/ascend-toolkit/latest"),
    ])

    for candidate in candidates:
        normalized = candidate
        if normalized.name in {"x86_64-linux", "aarch64-linux"}:
            normalized = normalized.parent
        set_env = normalized / "set_env.sh"
        if set_env.exists():
            return normalized.resolve(), set_env.resolve()

    raise RuntimeError("未找到 CANN Toolkit，请确认已安装 CANN，并设置环境变量。")


def source_cann_env(set_env):
    command = f"set -a && source {shlex.quote(str(set_env))} >/dev/null 2>&1 && env"
    result = subprocess.run(["bash", "-lc", command], check=True, text=True, capture_output=True)
    for line in result.stdout.splitlines():
        if "=" in line:
            key, value = line.split("=", 1)
            os.environ[key] = value


cann_home, cann_set_env = find_cann_home()
source_cann_env(cann_set_env)

WORKSPACE = Path("Sources/06.05")
WORKSPACE.mkdir(parents=True, exist_ok=True)

print(f"CANN Toolkit: {cann_home}")
print(f"Workspace: {WORKSPACE.resolve()}")

os.environ["ASCEND_HOME_PATH"] = str(cann_home)
os.environ["ASCEND_OP_COMPILE_SAVE_KERNEL_META"] = "1"
DEVICE_ID = 0  # 按实际空闲设备修改

---

准备源码工作目录：

本节的基础源码统一保存在 `src/06.05_superkernel_homework/01_superkernel_scope/`，其中包括网络脚本、运行脚本、GELU 与 `clear_ops` 算子源码、PyTorch 注册文件及构建脚本。

执行下面的单元格，将基础源码复制到 `Sources` 工作目录。

后续在 `Sources` 工作目录中进行内容修改。


In [ ]:
import shutil
from pathlib import Path

SRC_ROOT = Path("src/06.05_superkernel_homework/01_superkernel_scope")
DST_ROOT = Path("Sources/06.05/superkernel_question3")
ANSWER_ROOT = Path("answer/06.05_superkernel_homework")
assert SRC_ROOT.is_dir(), "请在 06_advanced_features 目录运行 notebook"
if not DST_ROOT.exists():
    shutil.copytree(SRC_ROOT, DST_ROOT, ignore=shutil.ignore_patterns(
        "build", "__pycache__", "*.pyc", "*.so", "*.egg-info", "profiling", "sk_meta"))
    print("已复制基础工程")
else:
    print("复用已有工作目录，保留练习修改")

OPERATOR = DST_ROOT / "ops/aclgraph_clear_ops/csrc/gelu_custom.asc"
NETWORK = DST_ROOT / "sample_network_segment.py"
print(f"算子源码：{OPERATOR.resolve()}")
print(f"网络脚本：{NETWORK.resolve()}")
print("工作目录文件：")
for path in sorted(DST_ROOT.rglob("*")):
    if path.is_file():
        print(path.relative_to(DST_ROOT))

---

`gelu_custom.asc` 代码补全：

在 `/Sources/06.05/superkernel_question3/ops/aclgraph_clear_ops/csrc/gelu_custom.asc` 中补全 TODO：新增 `GeluCustomKernelArgs`、`__sk__ __vector__` 模板入口 `gelu_custom_kernel_sk` 和 `SK_BIND`。结构体字段对应普通入口的 `(GM_ADDR x, GM_ADDR y)`，SK 入口读取地址并复用 `RunGelu()`；`splitNum` 不影响计算逻辑。

保留普通入口、shape/dtype/contiguous 检查及 PyTorch 注册。

---

`sample_network_segment.py` 代码补全：

在 `/Sources/06.05/superkernel_question3/sample_network_segment.py` 中完成修改：在 `forward` 函数中， `dequant_swiglu_quant` 后、`grouped_matmul_after` 前调用 `torch.ops.ascendc_ops.gelu_custom(x)`。

编译配置使用 `backend="npugraph_ex"`、`fullgraph=True`、`dynamic=False`，设置 `static_kernel_compile=True` 和 `super_kernel_optimize=True`。修改后保存文件。

---

执行以下命令进行编译并验证运行：

沿用工程的 `run_q1.sh` 构建扩展并运行网络。虽然文件名含 q1，本题仍使用该入口。


In [ ]:
subprocess.run([
    "bash", str((DST_ROOT / "run_q1.sh").resolve()),
    "--device-id", str(DEVICE_ID), "--warmup", "1", "--steps", "1",
], check=True)

预期执行效果如下（用于说明检查项，不是本 notebook 的实测记录）：

```text
warmup complete: 1 step(s)
profile step complete: 1/1
output: dtype=torch.int8, shape=(9216, 7168)
```

现有样例只打印 dtype/shape，**不会自动输出 allclose 或正确性结论**。提交结果时还需补充数值验证：

1. 固定输入与权重，在相同软件环境、设备和随机种子下运行包含 GELU 的 SK 关闭、开启版本，保存并比较 dtype、shape 和数值；不要比较无 GELU 与有 GELU 的网络。
2. 记录比较方法、误差指标和通过标准。若出现差异，单独验证普通直调 GELU，参考计算需匹配源码的近似公式和 int8 转换规则。
3. 数值对照与正式性能采集分开，避免比较操作影响 Profiling。


---

采集 Profiling：

先将网络脚本中的 `super_kernel_optimize` 设为 `False` 并保存，再执行以下单元格采集 baseline。命令不会自动修改开关，请先检查源码。


In [ ]:
from datetime import datetime

PROFILE_ROOT = (DST_ROOT / "profiling" / datetime.now().strftime("%Y%m%d_%H%M%S")).resolve()
print(NETWORK.read_text(encoding="utf-8"))
subprocess.run([
    "bash", str((DST_ROOT / "run_q1.sh").resolve()),
    "--device-id", str(DEVICE_ID), "--warmup", "1", "--steps", "1",
    "--profiler", "--output", str(PROFILE_ROOT / "baseline"),
], check=True)

将同一脚本的 `super_kernel_optimize` 改回 `True` 并保存，再采集 SK 版本。其余网络结构、设备及运行参数保持一致。


In [ ]:
subprocess.run([
    "bash", str((DST_ROOT / "run_q1.sh").resolve()),
    "--device-id", str(DEVICE_ID), "--warmup", "1", "--steps", "1",
    "--profiler", "--output", str(PROFILE_ROOT / "superkernel"),
], check=True)
print(f"Profiling 目录：{PROFILE_ROOT}")

结合两份 Profiling 和融合编译产物，判断 GELU 是否参与 SuperKernel 融合，比较任务数量和耗时，并说明性能变化。

开启 sk 前后的 profiling 对比分析可以参考[sk performance anlysis](answer/06.05_superkernel_homework/sk_performance_analysis.md)。

---

本次实践完成 GELU 的 SK 适配、网络接入与融合使能。请整理修改后的算子和网络代码，以及正确性验证、融合与性能分析结果。

执行下面的单元格可以打印参考答案。


In [ ]:
print((ANSWER_ROOT / "gelu_custom.asc").read_text(encoding="utf-8"))
print((ANSWER_ROOT / "sample_network_segment_with_gelu.py").read_text(encoding="utf-8"))

参考答案统一放在本节最后，仅用于对照，不参与前面的练习流程。完成自己的修改并验证后，再执行下面的单元格查看答案源码。

In [ ]:
REFERENCE_ROOT = Path("Sources/06.05/question3_reference")
if not REFERENCE_ROOT.exists():
    shutil.copytree(SRC_ROOT, REFERENCE_ROOT, ignore=shutil.ignore_patterns(
        "build", "__pycache__", "*.pyc", "*.so", "*.egg-info", "profiling", "sk_meta"))
shutil.copy2(ANSWER_ROOT / "gelu_custom.asc",
             REFERENCE_ROOT / "ops/aclgraph_clear_ops/csrc/gelu_custom.asc")
shutil.copy2(ANSWER_ROOT / "sample_network_segment_with_gelu.py",
             REFERENCE_ROOT / "sample_network_segment.py")
subprocess.run([
    "bash", str((REFERENCE_ROOT / "run_q1.sh").resolve()),
    "--device-id", str(DEVICE_ID), "--warmup", "1", "--steps", "1",
], check=True)